# Contact map setup

Generates the DNA inter-strand contact map used as a PLUMED CONTACTMAP restraint in metadynamics simulations.

**Workflow:**
1. Load the post-`editconf` structure (`_ed.gro`) as the reference
2. Identify all heavy-atom pairs within the DNA (inter-residue, separation > 3 residues) closer than `cutoff`
3. Write a text file (`_cmap.txt`) with atom index pairs and reference distances
4. Convert to PLUMED CONTACTMAP format (`_cmap.dat`)
5. Verify Q ≈ 1 on the reference structure

**Note:** The `_ed.gro` file (output of `gmx editconf`) is used — not the solvated/ionised structure — so atom indices match the unsolvated topology used in `pdb2gmx`.

In [1]:
# ── System parameters ──────────────────────────────────────────────────────
pdb = "1T63D_DNA"
ff  = "parambsc1"
wat = "tip3p"

ref_gro  = f"data/{pdb}_{ff}_{wat}_ed.gro"
cmap_txt = f"{pdb}_{ff}_cmap.txt"   # text: atom1 atom2 ref_dist
cmap_dat = f"{pdb}_{ff}_cmap.dat"   # PLUMED CONTACTMAP block

print(f"Reference structure : {ref_gro}")
print(f"Contact map (text)  : {cmap_txt}")
print(f"Contact map (PLUMED): {cmap_dat}")

Reference structure : data/1T63D_DNA_parambsc1_tip3p_ed.gro
Contact map (text)  : 1T63D_DNA_parambsc1_cmap.txt
Contact map (PLUMED): 1T63D_DNA_parambsc1_cmap.dat


In [2]:
import numpy as np
import mdtraj as md
from itertools import combinations

def best_hummer_q(traj, native, selection=None, cutoff=0.45, lambda_const=1.8, cmap=False):
    """Compute the fraction of native contacts according the definition from
    Best, Hummer and Eaton [1]
    
    Parameters
    ----------
    traj : md.Trajectory
        The trajectory to do the computation for
    native : md.Trajectory
        The 'native state'. This can be an entire trajecory, or just a single frame.
        Only the first conformation is used
    cutoff : float
        The cutoff for native contacts
    lambda_const : float
        Constant for the shift in the sigmoidal function
    selection : list
        Initial and final residues for Q calculation
    cmap : bool
        Whether to write the contact map to file or not
        
    Returns
    -------
    q : np.array, shape=(len(traj),)
        The fraction of native contacts in each frame of `traj`
        
    References
    ----------
    ..[1] Best, Hummer, and Eaton, "Native contacts determine protein folding
          mechanisms in atomistic simulations" PNAS (2013)
    """
    
    BETA_CONST = 50  # 1/nm
    LAMBDA_CONST = lambda_const
    NATIVE_CUTOFF = cutoff  # nanometers
    
    # get the indices of all of the heavy atoms
    if not selection:
        heavy = native.topology.select_atom_indices('heavy')
    else:
        try:
            print (selection[0], selection[1])
            heavy = native.top.select(f"index {selection[0]} to {selection[1]}")
        except Exception as e:
            print (e)

    # get the pairs of heavy atoms which are farther than 3
    # residues apart
    heavy_pairs = np.array(
        [(i,j) for (i,j) in combinations(heavy, 2)
            if abs(native.topology.atom(i).residue.index - \
                   native.topology.atom(j).residue.index) > 3])
        
    # compute the distances between these pairs in the native state
    heavy_pairs_distances = md.compute_distances(native[0], heavy_pairs)[0]
    
    # and get the pairs s.t. the distance is less than NATIVE_CUTOFF
    native_contacts = heavy_pairs[heavy_pairs_distances < NATIVE_CUTOFF]
    native_distances = heavy_pairs_distances[heavy_pairs_distances < NATIVE_CUTOFF]
    print("Number of native contacts", len(native_contacts))
    if cmap:
        np.savetxt(cmap, np.column_stack([native_contacts,native_distances]), \
                fmt=' %8i %8i %10.2f')
    else:
        # now compute these distances for the whole trajectory
        r = md.compute_distances(traj, native_contacts)
        # and recompute them for just the native state
        r0 = md.compute_distances(native[0], native_contacts)
    
        q = np.mean(1.0 / (1 + np.exp(BETA_CONST * (r - LAMBDA_CONST * r0))), axis=1)
        return q  

In [3]:
# ── Load reference structure and identify DNA atom range ───────────────────
top = md.load(ref_gro)

dna_atoms = top.topology.select("resname DG DA DC DT")
sel_start = int(dna_atoms[0])
sel_end   = int(dna_atoms[-1])

print(f"Total atoms in system : {top.n_atoms}")
print(f"DNA atoms (0-indexed) : {sel_start} to {sel_end}  ({len(dna_atoms)} atoms)")

Total atoms in system : 2487
DNA atoms (0-indexed) : 1537 to 2486  (950 atoms)


In [4]:
# ── Generate text contact map ──────────────────────────────────────────────
# cutoff=0.3 nm and lambda_const=1.4 match the WT 1Nhp6A parametrisation
best_hummer_q(top, top[0],
              selection=[sel_start, sel_end],
              cutoff=0.3, cmap=cmap_txt, lambda_const=1.4)

print(f"Contact map written to {cmap_txt}")

1537 2486
Number of native contacts 141
Contact map written to 1T63D_DNA_parambsc1_cmap.txt


In [5]:
%%bash -s "$cmap_txt" "$cmap_dat"
cmap_txt=$1
cmap_dat=$2

nlines=$(wc -l < "$cmap_txt")
echo "Contacts: $nlines"

awk -v nlines=$nlines \
    'BEGIN {print "cmap: CONTACTMAP ..."} \
     {print "   ATOMS"NR"="$1+1","$2+1" SWITCH"NR"={Q R_0=0.1 BETA=50.0 LAMBDA=1.4 REF="$3"} WEIGHT"NR"="1/nlines} \
     END {print "\n   SUM\n...\nPRINT ARG=cmap FILE=colvar"}' \
    "$cmap_txt" > "$cmap_dat"

echo "PLUMED file written: $cmap_dat"
head -4 "$cmap_dat"

Contacts: 141


PLUMED file written: 1T63D_DNA_parambsc1_cmap.dat


cmap: CONTACTMAP ...
   ATOMS1=1553,2477 SWITCH1={Q R_0=0.1 BETA=50.0 LAMBDA=1.4 REF=0.27} WEIGHT1=0

.0070922
   ATOMS2=1554,2475 SWITCH2={Q R_0=0.1 BETA=50.0 LAMBDA=1.4 REF=0.28} WEIGHT2=0.0070922
   

ATOMS3=1554,2477 SWITCH3={Q R_0=0.1 BETA=50.0 LAMBDA=1.4 REF=0.18} WEIGHT3=0.0070922


In [6]:
# ── Verification: Q on reference structure should be ~1.0 ─────────────────
q_ref = best_hummer_q(top, top[0],
                      selection=[sel_start, sel_end],
                      cutoff=0.3, lambda_const=1.4)
print(f"Q on reference structure: {q_ref[0]:.4f}  (expected ≈ 1.0)")

1537 2486


Number of native contacts 141
Q on reference structure: 0.9920  (expected ≈ 1.0)


## Next steps

1. **Embed cmap in PLUMED input:** paste the contents of `1T63D_DNA_parambsc1_cmap.dat` into `plumed_meta_2D_1T63D.dat`, replacing the `cmap:` and `dbias:` blocks from the WT file.

2. **Update atom indices for tail and DNA backbone CVs:** run
   ```bash
   python3 get_group_indices.py 1T63D_DNA_parambsc1.gro
   ```
   and update the `tail` and `dna_backbone` GROUP definitions in `plumed_meta_2D_1T63D.dat`.

3. **Update dna_center / dna_end1 / dna_end2 ranges:** DNA atoms in T63D run from index **1538–2487** (1-indexed, i.e. +1 from MDTraj 0-indexed 1537–2486). Subtract 2 from all DNA atom indices used in the WT PLUMED file, or re-derive them from the GRO.

4. **Calibrate `coord_tail_dna` baseline** on a short plain-MD run before launching the full metadynamics walkers.